In [1]:
from pathlib import Path
import polars as pl

BRONZE_DIR = Path("..") / "data" / "bronze"

train_tx = pl.scan_parquet(BRONZE_DIR / "train_transaction.parquet")
train_id = pl.scan_parquet(BRONZE_DIR / "train_identity.parquet")
test_tx = pl.scan_parquet(BRONZE_DIR / "test_transaction.parquet")
test_id = pl.scan_parquet(BRONZE_DIR / "test_identity.parquet")

In [2]:
# Fix the names of columns in test identity
test_id_fixed = test_id.rename(lambda col: col.replace("id-", "id_") if col.startswith("id-") else col)

In [3]:
test_id_fixed.collect_schema().names()

['TransactionID',
 'id_01',
 'id_02',
 'id_03',
 'id_04',
 'id_05',
 'id_06',
 'id_07',
 'id_08',
 'id_09',
 'id_10',
 'id_11',
 'id_12',
 'id_13',
 'id_14',
 'id_15',
 'id_16',
 'id_17',
 'id_18',
 'id_19',
 'id_20',
 'id_21',
 'id_22',
 'id_23',
 'id_24',
 'id_25',
 'id_26',
 'id_27',
 'id_28',
 'id_29',
 'id_30',
 'id_31',
 'id_32',
 'id_33',
 'id_34',
 'id_35',
 'id_36',
 'id_37',
 'id_38',
 'DeviceType',
 'DeviceInfo']

In [4]:
print(train_id.select(pl.col("TransactionID").is_duplicated().sum()).collect())
print(test_id_fixed.select(pl.col("TransactionID").is_duplicated().sum()).collect())
print(train_tx.select(pl.col("TransactionID").is_duplicated().sum()).collect())
print(test_tx.select(pl.col("TransactionID").is_duplicated().sum()).collect())

shape: (1, 1)
┌───────────────┐
│ TransactionID │
│ ---           │
│ u32           │
╞═══════════════╡
│ 0             │
└───────────────┘
shape: (1, 1)
┌───────────────┐
│ TransactionID │
│ ---           │
│ u32           │
╞═══════════════╡
│ 0             │
└───────────────┘
shape: (1, 1)
┌───────────────┐
│ TransactionID │
│ ---           │
│ u32           │
╞═══════════════╡
│ 0             │
└───────────────┘
shape: (1, 1)
┌───────────────┐
│ TransactionID │
│ ---           │
│ u32           │
╞═══════════════╡
│ 0             │
└───────────────┘


In [5]:
# train_transaction left-joined with train_identity on TransactionID, and the same for test
train = train_tx.join(train_id, on="TransactionID", how="left")
test = test_tx.join(test_id_fixed, on="TransactionID", how="left")

In [6]:
print(len(train.collect_schema().names()))
print(len(test.collect_schema().names()))

434
433


In [7]:
set(train.collect_schema().names()) - set(test.collect_schema().names())

{'isFraud'}

In [8]:
set(test.collect_schema().names()) - set(train.collect_schema().names())

set()

In [9]:
train.select(pl.len()).collect()

len
u32
590540


In [10]:
test.select(pl.len()).collect()

len
u32
506691
